# Laboratorio Calificado 02 - MiniMarket Normalizado

Notebook listo para Google Colab.

Este notebook usa la misma base de datos del LC2: `minimarket_normalizacion.db`.

Estructura del codigo:
- `pd.read_sql_query(...)` ejecuta la consulta SQL sobre la conexion `conn`.
- `display(df)` muestra el resultado como tabla en Google Colab.
- No se crean tablas temporales.
- No se cierra la conexion para poder seguir ejecutando consultas.


## 0. Descargar y conectar la base de datos


In [1]:
import sqlite3
import pandas as pd
import subprocess
from pathlib import Path

url = (
    'https://raw.githubusercontent.com/Rocio'
    'sayan/PMD1_Fundamentos_Gestion_Datos/main'
    '/casos/18_minimarket_normalizacion/minimarket_normalizacion.db'
)
db_path = Path('minimarket_normalizacion.db')

comando = ['curl', '--ssl-no-revoke', '-L', '--fail', '-o', str(db_path), url]
resultado = subprocess.run(comando, capture_output=True, text=True)

if resultado.returncode != 0:
    comando = ['curl', '-L', '--fail', '-o', str(db_path), url]
    resultado = subprocess.run(comando, capture_output=True, text=True)

if resultado.returncode != 0:
    raise RuntimeError(
        'No se pudo descargar la base de datos. Revisa tu conexion.'
    )

conn = sqlite3.connect(db_path)
print('Base de datos del minimarket cargada correctamente.')


Base de datos del minimarket cargada correctamente.


## Ejercicio 1. Inventario inicial de tablas del minimarket

Antes de responder preguntas del negocio, el equipo debe confirmar que la base de datos se cargo correctamente y reconocer que entidades existen.


In [2]:
# Ver las tablas disponibles
df_tablas = pd.read_sql_query(
    '''
    SELECT name AS tabla
    FROM sqlite_schema
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    ''',
    conn
)
display(df_tablas)


,tabla
0,categorias
1,clientes
2,locales
3,productos
4,ventas
5,ventas_original


## Ejercicio 2. Catalogo de productos con su categoria

El area comercial necesita mostrar el catalogo de productos junto con el nombre de su categoria para revisar precios y familias de productos.


In [3]:
# Productos con el nombre de su categoria
df_catalogo = pd.read_sql_query(
    '''
    SELECT
        p.id_producto,
        p.nombre_producto,
        cat.nombre_categoria,
        p.precio_unitario
    FROM productos p
    INNER JOIN categorias cat ON p.id_categoria = cat.id_categoria
    ORDER BY cat.nombre_categoria, p.nombre_producto
    ''',
    conn
)
display(df_catalogo)


,id_producto,nombre_producto,nombre_categoria,precio_unitario
0,2,Aceite Primor 1L,Abarrotes,12.8
1,1,Arroz Costeno 5kg,Abarrotes,28.5
2,3,Fideo Don Vittorio 500g,Abarrotes,4.2
3,5,Agua San Luis 2.5L,Bebidas,3.8
4,4,Gaseosa Inca Kola 3L,Bebidas,10.5
5,8,Jabon Protex,Cuidado personal,4.8
6,6,Detergente Bolivar 1kg,Limpieza,16.5
7,7,Lejia Clorox 1L,Limpieza,5.9


## Ejercicio 3. Boleta resumida de ventas recientes

El minimarket desea reconstruir una boleta resumida de las ventas recientes para responder consultas de clientes y revisar productos vendidos.


In [4]:
# Boleta resumida desde el 5 de abril de 2026
df_boleta = pd.read_sql_query(
    '''
    SELECT
        v.id_venta,
        v.fecha,
        c.nombre_cliente,
        c.distrito_cliente,
        p.nombre_producto,
        l.nombre_local,
        v.cantidad,
        p.precio_unitario,
        ROUND(v.cantidad * p.precio_unitario, 2) AS total_venta
    FROM ventas v
    INNER JOIN clientes c ON v.id_cliente = c.id_cliente
    INNER JOIN productos p ON v.id_producto = p.id_producto
    INNER JOIN locales l ON v.id_local = l.id_local
    WHERE v.fecha >= '2026-04-05'
    ORDER BY v.fecha, v.id_venta
    ''',
    conn
)
display(df_boleta)


,id_venta,fecha,nombre_cliente,distrito_cliente,nombre_producto,nombre_local,cantidad,precio_unitario,total_venta
0,9,2026-04-05,Jorge Medina,Los Olivos,Jabon Protex,MiniMarket Central,3,4.8,14.4
1,10,2026-04-05,Maria Perez,Independencia,Gaseosa Inca Kola 3L,MiniMarket Plaza,2,10.5,21.0
2,11,2026-04-06,Carlos Rios,San Martin,Detergente Bolivar 1kg,MiniMarket Norte,2,16.5,33.0
3,12,2026-04-06,Rosa Salazar,Puente Piedra,Aceite Primor 1L,MiniMarket Central,1,12.8,12.8


## Ejercicio 4. Distritos de clientes con menor actividad

Marketing quiere saber en que distritos hay clientes registrados, pero pocas compras, para decidir donde enviar promociones.


In [5]:
# Actividad comercial por distrito del cliente
df_distritos = pd.read_sql_query(
    '''
    SELECT
        c.distrito_cliente,
        COUNT(DISTINCT c.id_cliente) AS clientes_registrados,
        COUNT(v.id_venta) AS compras_registradas,
        ROUND(
            SUM(COALESCE(v.cantidad * p.precio_unitario, 0)),
            2
        ) AS ingresos_generados
    FROM clientes c
    LEFT JOIN ventas v ON c.id_cliente = v.id_cliente
    LEFT JOIN productos p ON v.id_producto = p.id_producto
    GROUP BY c.distrito_cliente
    ORDER BY ingresos_generados ASC
    ''',
    conn
)
display(df_distritos)


,distrito_cliente,clientes_registrados,compras_registradas,ingresos_generados
0,Puente Piedra,1,2,24.6
1,Independencia,1,2,37.5
2,San Martin,1,2,49.8
3,Comas,2,2,81.0
4,Los Olivos,2,4,132.6


## Ejercicio 5. Categorias mas fuertes por local

La administradora quiere saber que categorias generan mas ingresos en cada local para planificar reposicion de productos.


In [6]:
# Ingresos por local y categoria
df_local_categoria = pd.read_sql_query(
    '''
    SELECT
        l.nombre_local,
        cat.nombre_categoria,
        COUNT(v.id_venta) AS cantidad_ventas,
        SUM(v.cantidad) AS unidades_vendidas,
        ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingresos_totales
    FROM ventas v
    INNER JOIN locales l ON v.id_local = l.id_local
    INNER JOIN productos p ON v.id_producto = p.id_producto
    INNER JOIN categorias cat ON p.id_categoria = cat.id_categoria
    GROUP BY
        l.id_local, l.nombre_local,
        cat.id_categoria, cat.nombre_categoria
    ORDER BY l.nombre_local, ingresos_totales DESC
    ''',
    conn
)
display(df_local_categoria)


,nombre_local,nombre_categoria,cantidad_ventas,unidades_vendidas,ingresos_totales
0,MiniMarket Central,Abarrotes,3,6,108.2
1,MiniMarket Central,Bebidas,1,6,22.8
2,MiniMarket Central,Cuidado personal,1,3,14.4
3,MiniMarket Norte,Bebidas,1,5,52.5
4,MiniMarket Norte,Limpieza,2,4,44.8
5,MiniMarket Norte,Abarrotes,1,1,28.5
6,MiniMarket Plaza,Bebidas,1,2,21.0
7,MiniMarket Plaza,Abarrotes,1,4,16.8
8,MiniMarket Plaza,Limpieza,1,1,16.5


## Ejercicio 6. Categorias con ticket promedio relevante

Gerencia quiere revisar solo categorias que tienen varias ventas y cuyo ticket promedio supera un monto minimo.


In [7]:
# Categorias con al menos 3 ventas y ticket promedio mayor a 25 soles
df_ticket_categoria = pd.read_sql_query(
    '''
    SELECT
        cat.nombre_categoria,
        COUNT(v.id_venta) AS cantidad_ventas,
        ROUND(AVG(v.cantidad * p.precio_unitario), 2) AS ticket_promedio,
        ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingresos_totales
    FROM ventas v
    INNER JOIN productos p ON v.id_producto = p.id_producto
    INNER JOIN categorias cat ON p.id_categoria = cat.id_categoria
    GROUP BY cat.id_categoria, cat.nombre_categoria
    HAVING cantidad_ventas >= 3
       AND ticket_promedio > 25
    ORDER BY ticket_promedio DESC
    ''',
    conn
)
display(df_ticket_categoria)


,nombre_categoria,cantidad_ventas,ticket_promedio,ingresos_totales
0,Bebidas,3,32.1,96.3
1,Abarrotes,5,30.7,153.5


## Ejercicio 7. Dias con mayor movimiento de ventas

El encargado desea identificar dias con varias ventas para planificar mejor la atencion y reposicion diaria.


In [8]:
# Dias con dos o mas ventas registradas
df_dias_movimiento = pd.read_sql_query(
    '''
    SELECT
        fecha,
        COUNT(id_venta) AS cantidad_ventas,
        SUM(cantidad) AS unidades_vendidas
    FROM ventas
    GROUP BY fecha
    HAVING cantidad_ventas >= 2
    ORDER BY fecha
    ''',
    conn
)
display(df_dias_movimiento)


,fecha,cantidad_ventas,unidades_vendidas
0,2026-04-01,2,7
1,2026-04-02,2,4
2,2026-04-03,2,7
3,2026-04-04,2,6
4,2026-04-05,2,5
5,2026-04-06,2,3


## Ejercicio 8. Redundancia de productos en la tabla original

El equipo debe demostrar por que la tabla original desnormalizada genera repeticion de datos y posibles errores de actualizacion.


In [9]:
# Productos repetidos en la tabla original desnormalizada
df_redundancia_producto = pd.read_sql_query(
    '''
    SELECT
        producto,
        categoria,
        COUNT(*) AS veces_repetido,
        ROUND(SUM(total_venta), 2) AS ingresos_en_tabla_original
    FROM ventas_original
    GROUP BY producto, categoria
    HAVING veces_repetido >= 2
    ORDER BY veces_repetido DESC, ingresos_en_tabla_original DESC
    ''',
    conn
)
display(df_redundancia_producto)


,producto,categoria,veces_repetido,ingresos_en_tabla_original
0,Arroz Costeno 5kg,Abarrotes,2,85.5
1,Gaseosa Inca Kola 3L,Bebidas,2,73.5
2,Aceite Primor 1L,Abarrotes,2,51.2
3,Detergente Bolivar 1kg,Limpieza,2,49.5


## Ejercicio 9. Impacto de cambiar el nombre de una categoria

El minimarket piensa renombrar una categoria. El equipo debe comparar el impacto en la tabla original y en el modelo normalizado.


In [10]:
# Cuantas filas de ventas_original contienen cada categoria
df_categoria_original = pd.read_sql_query(
    '''
    SELECT
        categoria,
        COUNT(*) AS filas_afectadas_si_cambia_nombre
    FROM ventas_original
    GROUP BY categoria
    ORDER BY filas_afectadas_si_cambia_nombre DESC
    ''',
    conn
)
display(df_categoria_original)

# Productos asociados a cada categoria en el modelo normalizado
df_categoria_normalizada = pd.read_sql_query(
    '''
    SELECT
        cat.nombre_categoria,
        COUNT(p.id_producto) AS productos_asociados
    FROM categorias cat
    LEFT JOIN productos p ON cat.id_categoria = p.id_categoria
    GROUP BY cat.id_categoria, cat.nombre_categoria
    ORDER BY productos_asociados DESC
    ''',
    conn
)
display(df_categoria_normalizada)


,categoria,filas_afectadas_si_cambia_nombre
0,Abarrotes,5
1,Limpieza,3
2,Bebidas,3
3,Cuidado personal,1


,nombre_categoria,productos_asociados
0,Abarrotes,3
1,Bebidas,2
2,Limpieza,2
3,Cuidado personal,1


## Ejercicio 10. Clientes de mayor valor para fidelizacion

La gerencia quiere identificar clientes con monto acumulado mayor a 80 soles para ofrecerles una promocion o beneficio.


In [11]:
# Clientes con monto acumulado mayor a 80 soles
df_clientes_valor = pd.read_sql_query(
    '''
    SELECT
        c.nombre_cliente,
        c.distrito_cliente,
        COUNT(v.id_venta) AS cantidad_compras,
        SUM(v.cantidad) AS unidades_compradas,
        ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS monto_total
    FROM ventas v
    INNER JOIN clientes c ON v.id_cliente = c.id_cliente
    INNER JOIN productos p ON v.id_producto = p.id_producto
    GROUP BY c.id_cliente, c.nombre_cliente, c.distrito_cliente
    HAVING monto_total > 80
    ORDER BY monto_total DESC
    ''',
    conn
)
display(df_clientes_valor)


,nombre_cliente,distrito_cliente,cantidad_compras,unidades_compradas,monto_total
0,Ana Torres,Los Olivos,3,11,118.2
1,Luis Ramos,Comas,2,6,81.0
